# RMS Feature Extraction for Beehive Acoustic Monitoring

## Aim
This notebook extracts **Root Mean Square (RMS) energy** from preprocessed beehive recordings, aligns the resulting acoustic feature timeline with environmental logs, and visualises temporal activity patterns.

RMS is used here as a **proxy for relative acoustic energy**, which may reflect:
- collective bee movement
- wing vibration intensity
- periods of agitation or disturbance
- broader changes in colony activity over time

## Inputs
- preprocessed WAV files
- environmental log file (temperature and humidity)

## Outputs
- one RMS CSV per audio file
- one merged and aligned RMS dataset
- normalised version for visual comparison
- chunked plots for temporal inspection

## Notes
Because the recordings were preprocessed and may also have been normalised, RMS values should be interpreted as **relative energy changes**, not necessarily as absolute sound pressure levels.


## 1. Import libraries

In [ ]:
import os
from glob import glob
from datetime import timedelta

import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

## 2. Configuration

Update the paths and timestamps below for each recording set. Keeping all settings in one place makes the notebook easier to reuse across sessions and cleaner for presentation.


In [ ]:
# =========================
# USER CONFIGURATION
# =========================

# Audio and metadata paths
INPUT_FOLDER = r"X:\Dissertation Files\Set I\Preprocessed files Set I.II 13.08.2024 - 15.08.2024"
ENV_PATH = r"X:\Dissertation Files\Set I\Preprocessed files Set I.II 13.08.2024 - 15.08.2024\Env_logs\HaniBi_20240815T134229+0200.xlsx"

# Output folders
RMS_OUTPUT_FOLDER = os.path.join(INPUT_FOLDER, "RMS_features")
PLOT_OUTPUT_FOLDER = os.path.join(RMS_OUTPUT_FOLDER, "plots")

# Recording session timing
BASE_START_TIME = pd.Timestamp("2024-08-13 11:20:00")
START_SLICE = pd.Timestamp("2024-08-13 11:20:00")
END_SLICE = pd.Timestamp("2024-08-15 13:42:29")

# Feature extraction parameters
FRAME_LENGTH = 2048
HOP_LENGTH = 16000

# Plotting parameters
CHUNK_SECONDS = 3600  # 1 hour chunks

os.makedirs(RMS_OUTPUT_FOLDER, exist_ok=True)
os.makedirs(PLOT_OUTPUT_FOLDER, exist_ok=True)

## 3. Define helper functions

The extraction is written as reusable functions so the workflow is easier to maintain, validate, and later migrate into a Python module if needed.


In [ ]:
def extract_rms_with_timestamps(audio_path, start_time, frame_length=2048, hop_length=16000):
    """
    Extract RMS energy from a WAV file and return a timestamped DataFrame.

    Parameters
    ----------
    audio_path : str
        Path to the audio file.
    start_time : pandas.Timestamp
        Real-world start time of the recording.
    frame_length : int
        Window length used for RMS calculation.
    hop_length : int
        Hop size between consecutive RMS frames.

    Returns
    -------
    pandas.DataFrame
        DataFrame with columns: timestamp, rms
    """
    y, sr = librosa.load(audio_path, sr=None)

    rms = librosa.feature.rms(
        y=y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    times = librosa.frames_to_time(
        np.arange(len(rms)),
        sr=sr,
        hop_length=hop_length
    )

    timestamps = [start_time + timedelta(seconds=float(t)) for t in times]

    return pd.DataFrame({
        "timestamp": timestamps,
        "rms": rms
    })


def load_and_prepare_environmental_log(env_path, start_slice, end_slice):
    """
    Load environmental log, standardise timestamp column, and trim it
    to the recording interval of interest.
    """
    env_df = pd.read_excel(env_path)
    env_df = env_df.rename(columns={"Date": "timestamp"})
    env_df["timestamp"] = pd.to_datetime(env_df["timestamp"])

    env_df = env_df[(env_df["timestamp"] >= start_slice) & (env_df["timestamp"] <= end_slice)].copy()
    env_df = env_df.sort_values("timestamp").reset_index(drop=True)

    return env_df


def compute_sequential_start_times(file_list, base_start_time):
    """
    Compute start times for multiple consecutive WAV files by summing
    the duration of previous files.
    """
    durations = [librosa.get_duration(path=f) for f in file_list]

    start_times = []
    elapsed = 0.0

    for duration in durations:
        start_times.append(base_start_time + timedelta(seconds=elapsed))
        elapsed += duration

    return start_times, durations


def align_rms_with_environmental_log(rms_df, env_df):
    """
    Align RMS timeline with environmental measurements using nearest timestamp.
    """
    aligned = pd.merge_asof(
        rms_df.sort_values("timestamp"),
        env_df.sort_values("timestamp"),
        on="timestamp",
        direction="nearest"
    )
    return aligned


def normalise_columns(df, columns):
    """
    Min-max normalise selected columns for joint plotting.
    """
    scaler = MinMaxScaler()
    norm_values = scaler.fit_transform(df[columns])

    norm_df = df.copy()
    norm_df[[f"{col}_norm" for col in columns]] = norm_values
    return norm_df


def plot_rms_chunks(norm_df, output_dir, chunk_seconds=3600):
    """
    Plot normalised RMS, temperature, and humidity in fixed-length time chunks.
    """
    start_time = norm_df["timestamp"].iloc[0]
    norm_df = norm_df.copy()
    norm_df["elapsed_sec"] = (norm_df["timestamp"] - start_time).dt.total_seconds()

    max_time = norm_df["elapsed_sec"].max()
    num_chunks = int(np.ceil(max_time / chunk_seconds))

    for i in range(num_chunks):
        chunk_start = i * chunk_seconds
        chunk_end = (i + 1) * chunk_seconds

        chunk_df = norm_df[
            (norm_df["elapsed_sec"] >= chunk_start) &
            (norm_df["elapsed_sec"] < chunk_end)
        ]

        if chunk_df.empty:
            continue

        plt.figure(figsize=(14, 5))
        plt.plot(chunk_df["timestamp"], chunk_df["rms_norm"], label="RMS (normalised)")
        plt.plot(chunk_df["timestamp"], chunk_df["Temperatura (°C)_norm"], label="Temperature (normalised)")
        plt.plot(chunk_df["timestamp"], chunk_df["Wilgotność (%)_norm"], label="Humidity (normalised)")

        plt.title(f"Normalised RMS and Environmental Variables | Chunk {i + 1}")
        plt.xlabel("Timestamp")
        plt.ylabel("Normalised Value")
        plt.xticks(rotation=45)
        plt.legend()
        plt.tight_layout()

        output_path = os.path.join(output_dir, f"rms_chunk_{i+1:02d}.png")
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.show()

## 4. Load and inspect environmental data

In [ ]:
env_df = load_and_prepare_environmental_log(
    env_path=ENV_PATH,
    start_slice=START_SLICE,
    end_slice=END_SLICE
)

env_df.head()

## 5. Batch RMS extraction across all preprocessed recordings

This step processes every WAV file in the session folder, calculates RMS values, assigns real timestamps based on cumulative duration, and saves one CSV file per recording.


In [ ]:
file_list = sorted(glob(os.path.join(INPUT_FOLDER, "*.wav")))
start_times, durations = compute_sequential_start_times(file_list, BASE_START_TIME)

processing_log = []

for file_path, file_start_time, duration in zip(file_list, start_times, durations):
    file_name = os.path.basename(file_path)

    try:
        rms_df = extract_rms_with_timestamps(
            audio_path=file_path,
            start_time=file_start_time,
            frame_length=FRAME_LENGTH,
            hop_length=HOP_LENGTH
        )

        output_csv = os.path.join(
            RMS_OUTPUT_FOLDER,
            os.path.splitext(file_name)[0] + "_rms.csv"
        )
        rms_df.to_csv(output_csv, index=False)

        processing_log.append({
            "file": file_name,
            "status": "success",
            "duration_seconds": duration,
            "rows_written": len(rms_df),
            "output_csv": output_csv
        })

    except Exception as e:
        processing_log.append({
            "file": file_name,
            "status": "failed",
            "duration_seconds": duration,
            "rows_written": np.nan,
            "output_csv": None,
            "error": str(e)
        })

processing_log_df = pd.DataFrame(processing_log)
processing_log_df

## 6. Combine all RMS CSVs and align with environmental variables

In [ ]:
rms_files = sorted(glob(os.path.join(RMS_OUTPUT_FOLDER, "*_rms.csv")))

combined_rms_df = pd.concat(
    [pd.read_csv(f, parse_dates=["timestamp"]) for f in rms_files],
    ignore_index=True
)

aligned_df = align_rms_with_environmental_log(combined_rms_df, env_df)
aligned_df = aligned_df.dropna(subset=["rms", "Temperatura (°C)", "Wilgotność (%)"]).reset_index(drop=True)

aligned_df.head()

## 7. Save merged aligned dataset

In [ ]:
aligned_output_path = os.path.join(RMS_OUTPUT_FOLDER, "rms_aligned_with_environment.csv")
aligned_df.to_csv(aligned_output_path, index=False)

aligned_output_path

## 8. Normalise RMS and environmental variables for comparison

Normalisation is used here **only for visual comparison**. It does not change the original aligned dataset and should not be confused with physical amplitude calibration.


In [ ]:
norm_df = normalise_columns(
    aligned_df,
    columns=["rms", "Temperatura (°C)", "Wilgotność (%)"]
)

norm_output_path = os.path.join(RMS_OUTPUT_FOLDER, "rms_aligned_normalised.csv")
norm_df.to_csv(norm_output_path, index=False)

norm_df.head()

## 9. Plot full-session overview

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(norm_df["timestamp"], norm_df["rms_norm"], label="RMS (normalised)")
plt.plot(norm_df["timestamp"], norm_df["Temperatura (°C)_norm"], label="Temperature (normalised)")
plt.plot(norm_df["timestamp"], norm_df["Wilgotność (%)_norm"], label="Humidity (normalised)")

plt.title("Normalised RMS and Environmental Variables Over Full Session")
plt.xlabel("Timestamp")
plt.ylabel("Normalised Value")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 10. Plot session in smaller chunks

Chunked plots make subtle changes more visible than a single long-duration figure and are especially useful when inspecting day-night variation or short periods of disturbance.


In [ ]:
plot_rms_chunks(
    norm_df=norm_df,
    output_dir=PLOT_OUTPUT_FOLDER,
    chunk_seconds=CHUNK_SECONDS
)

## 11. Export processing log

In [ ]:
log_output_path = os.path.join(RMS_OUTPUT_FOLDER, "rms_processing_log.csv")
processing_log_df.to_csv(log_output_path, index=False)

processing_log_df

## 12. Interpretation notes

### Why RMS matters in this project
RMS provides a compact time-series representation of acoustic energy inside the hive. In the context of bee monitoring, this can be used to study changes in **relative colony activity** over time.

### What higher RMS may suggest
- stronger collective buzzing
- more movement within the hive
- agitation, disturbance, or thermal stress
- increased behavioural intensity during certain periods

### What lower RMS may suggest
- quieter periods
- reduced movement
- resting or less active intervals

### Important limitation
Because the recordings were preprocessed and potentially normalised, RMS should be interpreted as a **relative feature** rather than an absolute measure of loudness.

### Role in the wider dissertation pipeline
This feature supports:
- temporal trend analysis
- comparison with temperature and humidity
- anomaly inspection
- later integration with other acoustic descriptors such as MFCCs, spectral centroid, bandwidth, rolloff, and zero-crossing rate
